# 12. Correlation Analysis — Linear, Rank & Multicollinearity (VIF)

Correlation measures the statistical association between variables. Different types of variables and relationships demand specific correlation coefficients:

1. **Pearson Product-Moment ($r$)**: Measures strictly **linear** association between continuous variables. Sensitive to outliers and requires bivariate normality.
2. **Spearman Rank ($\rho$)**: Measures **monotonic** association (linear or non-linear curves). Operates on ranks, highly robust to outliers.
3. **Kendall Tau ($\tau$)**: Measures concordance vs discordance of pairs. Ideal for small sample sizes with tied ranks.
4. **Multicollinearity & VIF**: Diagnoses linear redundancies where one feature is predicted by a combination of other features.

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats

np.random.seed(42)
n = 1000

# 1. Linear relationship
x1 = np.random.uniform(10, 100, size=n)
y_linear = 2.5 * x1 + np.random.normal(0, 15, size=n)

# 2. Monotonic non-linear relationship (Exponential curve)
y_monotonic = np.exp(x1 / 25.0) + np.random.normal(0, 2, size=n)

# 3. Non-monotonic non-linear relationship (Quadratic parabola)
x_quad = np.random.uniform(-50, 50, size=n)
y_quad = (x_quad ** 2) + np.random.normal(0, 100, size=n)

print("Correlation Experiment Initialized.")

Correlation Experiment Initialized.

## 1. Comparing Pearson vs. Spearman on Linear and Non-Linear Data

In [1]:
def compare_correlations(x, y, name):
    r_p, p_p = stats.pearsonr(x, y)
    r_s, p_s = stats.spearmanr(x, y)
    tau, p_t = stats.kendalltau(x, y)
    
    print(f"=== {name.upper()} RELATIONSHIP ===")
    print(f"  Pearson r:    {r_p:+.3f} (Linear Strength)")
    print(f"  Spearman rho: {r_s:+.3f} (Monotonic Strength)")
    print(f"  Kendall tau:  {tau:+.3f} (Rank Concordance)\n")

compare_correlations(x1, y_linear, 'Strictly Linear')
compare_correlations(x1, y_monotonic, 'Monotonic Non-Linear (Exp)')
compare_correlations(x_quad, y_quad, 'Parabolic Non-Monotonic')

=== STRICTLY LINEAR RELATIONSHIP ===
  Pearson r:    +0.974 (Linear Strength)
  Spearman rho: +0.973 (Monotonic Strength)
  Kendall tau:  +0.865 (Rank Concordance)

=== MONOTONIC NON-LINEAR (EXP) RELATIONSHIP ===
  Pearson r:    +0.841 (Linear Strength)
  Spearman rho: +0.996 (Monotonic Strength)
  Kendall tau:  +0.942 (Rank Concordance)

=== PARABOLIC NON-MONOTONIC RELATIONSHIP ===
  Pearson r:    -0.012 (Linear Strength)
  Spearman rho: -0.014 (Monotonic Strength)
  Kendall tau:  -0.009 (Rank Concordance)

## 2. Multicollinearity & Variance Inflation Factor (VIF)

Multicollinearity occurs when independent variables are highly correlated with each other. This inflates standard errors of regression coefficients, making them unstable.

$$VIF_i = \frac{1}{1 - R_i^2}$$
- $VIF = 1$: No collinearity.
- $1 < VIF < 5$: Moderate, acceptable collinearity.
- $VIF > 5-10$: High collinearity; feature pruning or regularization is required.

In [1]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Synthesize collinear feature matrix
x_base = np.random.normal(50, 10, size=500)
f1 = x_base + np.random.normal(0, 2, size=500)
f2 = x_base * 1.8 + np.random.normal(0, 3, size=500)  # Heavily collinear with f1
f3 = np.random.normal(20, 5, size=500)                 # Independent

df_vif = pd.DataFrame({'f1_collinear': f1, 'f2_collinear': f2, 'f3_independent': f3})
df_vif['intercept'] = 1.0

vif_data = pd.DataFrame()
vif_data["Feature"] = df_vif.columns
vif_data["VIF"] = [variance_inflation_factor(df_vif.values, i) for i in range(df_vif.shape[1])]

print("Multicollinearity VIF Audit:")
print(vif_data[vif_data['Feature'] != 'intercept'].sort_values(by='VIF', ascending=False))

Multicollinearity VIF Audit:
          Feature    VIF
0    f1_collinear 24.812
1    f2_collinear 24.810
2  f3_independent  1.012